In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import re
import nltk
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
df = pd.read_csv('fake_news_dataset.csv')

In [3]:
df.head()

,id,title,author,text,state,date_published,source,category,sentiment_score,word_count,...,num_shares,num_comments,political_bias,fact_check_rating,is_satirical,trust_score,source_reputation,clickbait_score,plagiarism_score,label
0,1,Breaking News 1,Jane Smith,This is the content of article 1. It contains ...,Tennessee,30-11-2021,The Onion,Entertainment,-0.22,1302,...,47305,450,Center,FALSE,1,76,6,0.84,53.35,Fake
1,2,Breaking News 2,Emily Davis,This is the content of article 2. It contains ...,Wisconsin,02-09-2021,The Guardian,Technology,0.92,322,...,39804,530,Left,Mixed,1,1,5,0.85,28.28,Fake
2,3,Breaking News 3,John Doe,This is the content of article 3. It contains ...,Missouri,13-04-2021,New York Times,Sports,0.25,228,...,45860,763,Center,Mixed,0,57,1,0.72,0.38,Fake
3,4,Breaking News 4,Alex Johnson,This is the content of article 4. It contains ...,North Carolina,08-03-2020,CNN,Sports,0.94,155,...,34222,945,Center,TRUE,1,18,10,0.92,32.20,Fake
4,5,Breaking News 5,Emily Davis,This is the content of article 5. It contains ...,California,23-03-2022,Daily Mail,Technology,-0.01,962,...,35934,433,Right,Mixed,0,95,6,0.66,77.70,Real


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 4000 non-null   int64  
 1   title              4000 non-null   str    
 2   author             4000 non-null   str    
 3   text               4000 non-null   str    
 4   state              4000 non-null   str    
 5   date_published     4000 non-null   str    
 6   source             4000 non-null   str    
 7   category           4000 non-null   str    
 8   sentiment_score    4000 non-null   float64
 9   word_count         4000 non-null   int64  
 10  char_count         4000 non-null   int64  
 11  has_images         4000 non-null   int64  
 12  has_videos         4000 non-null   int64  
 13  readability_score  4000 non-null   float64
 14  num_shares         4000 non-null   int64  
 15  num_comments       4000 non-null   int64  
 16  political_bias     4000 non-null   

In [5]:
df['label'].value_counts()

label
Fake    2026
Real    1974
Name: count, dtype: int64

In [6]:
df[['text', 'label']].isnull().sum()

text     0
label    0
dtype: int64

In [8]:
nltk.download('stopwords',quiet = True)
stop_words = set(stopwords.words('english'))

In [17]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return ' '.join([word for word in text.split() if word not in stop_words])

In [25]:
df['cleaned_text'] = df['text'].apply(clean_text)

In [26]:
df['label_clean'] = df['label'].astype(str).str.strip().str.upper()
df['binary_label'] = df['label_clean'].map({'REAL': 1, 'FAKE': 0})

In [27]:
df = df.dropna(subset=['binary_label', 'cleaned_text'])

In [28]:
texts = df['cleaned_text'].tolist()
labels = df['binary_label'].astype(int).tolist()

In [29]:
X_train_text, X_test_text, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

In [30]:
y_train = np.array(y_train, dtype=np.int32)
y_test = np.array(y_test, dtype=np.int32)

In [31]:
vocab_size = 10000
max_length = 500

In [32]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

In [33]:
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

In [34]:
X_train = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_test = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

In [36]:
X_train.shape

(3200, 500)

In [37]:
y_train.shape

(3200,)

In [38]:
X_test.shape

(800, 500)

In [39]:
y_test.shape

(800,)

In [41]:
from gensim.models import Word2Vec

In [42]:
w2v_data = [text.split() for text in X_train_text]
embedding_dim = 100

w2v_model = Word2Vec(sentences=w2v_data, 
                     vector_size=embedding_dim, 
                     window=5, 
                     min_count=2,
                     workers=4)

In [43]:
word_index = tokenizer.word_index
num_words = min(vocab_size, len(word_index) + 1)

In [44]:
embedding_matrix = np.zeros((num_words, embedding_dim))

In [45]:
missing_words = 0
for word, i in word_index.items():
    if i >= num_words:
        continue
    
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]
    else:
        missing_words += 1

In [46]:
embedding_matrix.shape

(8, 100)

In [56]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout

In [57]:
model = Sequential()
model.add(Input(shape=(max_length,)))

model.add(Embedding(input_dim=num_words, 
                    output_dim=embedding_dim, 
                    weights=[embedding_matrix], 
                    input_length=max_length, 
                    trainable=False))

In [58]:
model.add(LSTM(units=64))

In [59]:
model.add(Dropout(0.2))

In [60]:
model.add(Dense(units=32, activation='relu'))

In [61]:
model.add(Dense(units=1, activation='sigmoid'))

In [62]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ (None, 500, 100)            │             800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 64)                  │          42,240 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 45,153 (176.38 KB)

 Trainable params: 44,353 (173.25 KB)

 Non-trainable params: 800 (3.12 KB)

In [63]:
from tensorflow.keras.callbacks import EarlyStopping

model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

In [64]:
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

In [65]:
history = model.fit(
    X_train, y_train,
    epochs=10, 
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 19s 282ms/step - accuracy: 0.4909 - loss: 0.6932 - val_accuracy: 0.5138 - val_loss: 0.6931
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 214ms/step - accuracy: 0.5047 - loss: 0.6931 - val_accuracy: 0.5138 - val_loss: 0.6931
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 226ms/step - accuracy: 0.5047 - loss: 0.6931 - val_accuracy: 0.5138 - val_loss: 0.6930
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 223ms/step - accuracy: 0.5047 - loss: 0.6931 - val_accuracy: 0.5138 - val_loss: 0.6930
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 211ms/step - accuracy: 0.5047 - loss: 0.6931 - val_accuracy: 0.5138 - val_loss: 0.6930
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 216ms/step - accuracy: 0.5047 - loss: 0.6931 - val_accuracy: 0.5138 - val_loss: 0.6930
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 218ms/step - accuracy: 0.5047 - loss: 0.6931 - val_accuracy: 0.5138 - val_loss: 0.6930


In [66]:
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype(int)

25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step


In [67]:
print(confusion_matrix(y_test, y_pred))

[[411   0]
 [389   0]]


In [68]:
print(classification_report(y_test, y_pred, target_names=['FAKE (0)', 'REAL (1)']))

              precision    recall  f1-score   support

    FAKE (0)       0.51      1.00      0.68       411
    REAL (1)       0.00      0.00      0.00       389

    accuracy                           0.51       800
   macro avg       0.26      0.50      0.34       800
weighted avg       0.26      0.51      0.35       800



C:\Users\aeman\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\aeman\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\aeman\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [69]:
def predict_news(article_text):
    cleaned = clean_text(article_text)
    sequence = tokenizer.texts_to_sequences([cleaned])
    padded_sequence = pad_sequences(sequence, maxlen=max_length, padding='post', truncating='post')
    prediction_prob = model.predict(padded_sequence)[0][0]
    if prediction_prob > 0.5:
        print(f"\nPrediction: REAL NEWS")
    else:
        print(f"\nPrediction: FAKE NEWS")
    print(f"Confidence Score: {prediction_prob:.4f} (Closer to 1 is Real, Closer to 0 is Fake)")

In [70]:
article_1 = """
The government announced today that a new policy will be implemented next month to reduce taxes for middle-class families. 
The bill was passed with bipartisan support in the senate.
"""

article_2 = """
SHOCKING! Aliens have officially landed in New York City! The mayor has surrendered the city to the Martian overlords. 
Share this before it gets deleted by the mainstream media!
"""

print("\nTesting Article 1 (Looks like standard news):")
predict_news(article_1)

print("\nTesting Article 2 (Looks highly suspicious):")
predict_news(article_2)


Testing Article 1 (Looks like standard news):
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step

Prediction: FAKE NEWS
Confidence Score: 0.4965 (Closer to 1 is Real, Closer to 0 is Fake)

Testing Article 2 (Looks highly suspicious):
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step

Prediction: FAKE NEWS
Confidence Score: 0.4965 (Closer to 1 is Real, Closer to 0 is Fake)
